# Neural Jigsaw Reconstruction for Eroded 3×3 STL-10 Patches

This notebook reconstructs a complete RGB image of shape **96×96×3** from **9 scrambled eroded patches** of shape **28×28×3**.

The solution follows the assignment constraints:

- Keras/TensorFlow implementation.
- No pretrained models.
- Neural-only reconstruction pipeline.
- Trainable parameter budget below **6 million**.
- MAE is used as the final evaluation metric.
- The notebook includes a `gdown` loading cell for published weights and a save cell for producing the weights file.

Architecture summary:

1. A shared CNN encodes each patch.
2. A small self-attention network predicts where each patch belongs in the 3×3 grid.
3. A differentiable neural placement layer creates an ordered canvas with missing eroded borders.
4. A compact convolutional inpainting network fills the missing border pixels.
5. The known patch pixels are preserved by a neural blend mask, while only the missing regions are synthesized.


In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers
from keras.utils import PyDataset

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

# Reproducibility.
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 1. Download and load STL-10

The original notebook uses the STL-10 unlabeled set. This loader is written to be robust to minor differences in the extraction folder name used by `tf.keras.utils.get_file`.


In [ ]:
def download_and_load_stl10():
    path = tf.keras.utils.get_file(
        "stl10_binary.tar.gz",
        origin="https://ai.stanford.edu/~acoates/stl10/stl10_binary.tar.gz",
        extract=True,
    )

    base_dir = os.path.dirname(path)
    matches = glob.glob(os.path.join(base_dir, "**", "unlabeled_X.bin"), recursive=True)
    if not matches:
        raise FileNotFoundError(
            "Could not find unlabeled_X.bin after extraction. "
            f"Looked under: {base_dir}"
        )

    filepath = matches[0]
    print("Loading data from:", filepath)

    with open(filepath, "rb") as f:
        data = np.fromfile(f, dtype=np.uint8)

    # Same orientation convention as the provided specification notebook.
    images = np.reshape(data, (-1, 3, 96, 96))
    images = np.transpose(images, (0, 3, 2, 1))
    return images


images = download_and_load_stl10()
print("Loaded images:", images.shape, images.dtype)


In [ ]:
train_images = images[:80000]
val_images = images[80000:90000]
test_images = images[90000:]

print("Train:", train_images.shape)
print("Validation:", val_images.shape)
print("Test:", test_images.shape)


## 2. Patch generator with auxiliary placement labels

For training only, the generator also returns the original 3×3 position of each scrambled patch.  
This is not a hand-coded reconstruction algorithm; it is an auxiliary neural supervision signal that helps the placement head learn the jigsaw task.

At inference/evaluation time, the model receives only the scrambled patches.


In [ ]:
class PatchGeneratorWithLabels(PyDataset):
    def __init__(
        self,
        images,
        batch_size=64,
        patch_size=32,
        crop_size=28,
        shuffle=True,
        return_position_labels=True,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.images = images.astype("float32") / 255.0
        self.batch_size = batch_size
        self.patch_size = patch_size
        self.crop_size = crop_size
        self.shuffle = shuffle
        self.return_position_labels = return_position_labels
        self.indices = np.arange(len(self.images))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.images) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        bsz = len(batch_indices)

        x = np.zeros((bsz, 9, self.crop_size, self.crop_size, 3), dtype="float32")
        y_img = np.zeros((bsz, 96, 96, 3), dtype="float32")
        y_pos = np.zeros((bsz, 9), dtype="int32")

        margin = (self.patch_size - self.crop_size) // 2

        for i, img_idx in enumerate(batch_indices):
            full_img = self.images[img_idx]
            y_img[i] = full_img

            patches = []
            for r in range(3):
                for c in range(3):
                    y0 = r * self.patch_size
                    x0 = c * self.patch_size
                    patch = full_img[
                        y0 + margin : y0 + margin + self.crop_size,
                        x0 + margin : x0 + margin + self.crop_size,
                        :,
                    ]
                    patches.append(patch)

            # order[slot_idx] is the original grid position of the patch placed in input slot slot_idx.
            order = np.random.permutation(9)
            for slot_idx, original_pos in enumerate(order):
                x[i, slot_idx] = patches[original_pos]
                y_pos[i, slot_idx] = original_pos

        if self.return_position_labels:
            return x, {"recon": y_img, "assign": y_pos}
        return x, y_img

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


BATCH_SIZE = 64

train_generator = PatchGeneratorWithLabels(
    train_images, batch_size=BATCH_SIZE, shuffle=True, return_position_labels=True
)
val_generator = PatchGeneratorWithLabels(
    val_images, batch_size=BATCH_SIZE, shuffle=False, return_position_labels=True
)
test_generator = PatchGeneratorWithLabels(
    test_images, batch_size=BATCH_SIZE, shuffle=False, return_position_labels=False
)


In [ ]:
def plot_puzzle(patches, ordering=None):
    if ordering is None:
        order = np.arange(9)
    else:
        order = np.array(ordering).flatten()

    canvas = np.zeros((96, 96, 3), dtype=np.float32)
    cell_dim = 32
    patch_dim = 28
    margin = (cell_dim - patch_dim) // 2

    for i in range(9):
        grid_pos = order[i]
        row = grid_pos // 3
        col = grid_pos % 3
        y_start = row * cell_dim + margin
        x_start = col * cell_dim + margin
        canvas[y_start : y_start + patch_dim, x_start : x_start + patch_dim] = patches[i]

    plt.figure(figsize=(4, 4))
    plt.imshow(np.clip(canvas, 0, 1))
    plt.axis("off")
    plt.show()


x_batch, y_batch = train_generator[0]
plt.figure(figsize=(4, 4))
plt.imshow(y_batch["recon"][0])
plt.title("Target image")
plt.axis("off")
plt.show()

plot_puzzle(x_batch[0])


## 3. Model

The model has two outputs during training:

- `recon`: reconstructed 96×96×3 image.
- `assign`: patch-to-grid-position probability distribution, used only as auxiliary supervision.

The final metric is computed on `recon` only.


In [ ]:
SOFTMAX_TEMPERATURE = 0.50


def place_slots_on_96_canvas(slots):
    """Convert ordered slots of shape (B, 9, 28, 28, C) to a 96×96 canvas.

    Each 28×28 patch is placed in the center of its 32×32 cell.
    The 2-pixel border on each side remains missing and is filled by the inpainting CNN.
    """
    b = tf.shape(slots)[0]
    c = tf.shape(slots)[-1]

    grid = tf.reshape(slots, (b, 3, 3, 28, 28, c))
    grid = tf.pad(
        grid,
        paddings=[[0, 0], [0, 0], [0, 0], [2, 2], [2, 2], [0, 0]],
        mode="CONSTANT",
        constant_values=0.0,
    )
    grid = tf.transpose(grid, (0, 1, 3, 2, 4, 5))
    canvas = tf.reshape(grid, (b, 96, 96, c))
    return canvas


def build_jigsaw_reconstructor(
    embed_dim=192,
    transformer_blocks=4,
    heads=4,
    decoder_channels=64,
    decoder_blocks=8,
):
    patches = keras.Input(shape=(9, 28, 28, 3), name="scrambled_patches")

    # Shared patch encoder.
    x = layers.TimeDistributed(
        layers.Conv2D(32, 3, padding="same", activation="swish"), name="enc_conv1"
    )(patches)
    x = layers.TimeDistributed(layers.MaxPooling2D(), name="enc_pool1")(x)

    x = layers.TimeDistributed(
        layers.Conv2D(64, 3, padding="same", activation="swish"), name="enc_conv2"
    )(x)
    x = layers.TimeDistributed(layers.MaxPooling2D(), name="enc_pool2")(x)

    x = layers.TimeDistributed(
        layers.Conv2D(128, 3, padding="same", activation="swish"), name="enc_conv3"
    )(x)
    x = layers.TimeDistributed(layers.GlobalAveragePooling2D(), name="enc_gap")(x)
    x = layers.Dense(embed_dim, activation="swish", name="patch_embedding")(x)

    # Permutation-equivariant patch reasoning.
    for i in range(transformer_blocks):
        h = layers.LayerNormalization(name=f"tr_{i}_ln1")(x)
        h = layers.MultiHeadAttention(
            num_heads=heads,
            key_dim=embed_dim // heads,
            dropout=0.10,
            name=f"tr_{i}_mha",
        )(h, h)
        x = layers.Add(name=f"tr_{i}_attn_add")([x, h])

        h = layers.LayerNormalization(name=f"tr_{i}_ln2")(x)
        h = layers.Dense(embed_dim * 2, activation="swish", name=f"tr_{i}_mlp1")(h)
        h = layers.Dropout(0.10, name=f"tr_{i}_drop")(h)
        h = layers.Dense(embed_dim, name=f"tr_{i}_mlp2")(h)
        x = layers.Add(name=f"tr_{i}_mlp_add")([x, h])

    assign_logits = layers.Dense(9, name="assign_logits")(x)
    assign = layers.Softmax(axis=-1, name="assign")(assign_logits)

    # Differentiable neural placement:
    # for each destination slot, choose a patch using a softmax over patches.
    slot_weights = layers.Lambda(
        lambda z: tf.nn.softmax(z / SOFTMAX_TEMPERATURE, axis=1),
        output_shape=(9, 9),
        name="slot_weights",
    )(assign_logits)

    ordered_slots = layers.Lambda(
        lambda z: tf.einsum("bpk,bphwc->bkhwc", z[0], z[1]),
        output_shape=(9, 28, 28, 3),
        name="soft_patch_placement",
    )([slot_weights, patches])

    known_canvas = layers.Lambda(
        place_slots_on_96_canvas,
        output_shape=(96, 96, 3),
        name="known_rgb_canvas",
    )(ordered_slots)

    mask_slots = layers.Lambda(
        lambda z: tf.ones_like(z[..., :1]),
        output_shape=(9, 28, 28, 1),
        name="known_slot_mask",
    )(ordered_slots)

    known_mask = layers.Lambda(
        place_slots_on_96_canvas,
        output_shape=(96, 96, 1),
        name="known_mask",
    )(mask_slots)

    # A compact convolutional inpainting network fills only missing eroded borders.
    d = layers.Concatenate(name="inpaint_input")([known_canvas, known_mask])
    d = layers.Conv2D(
        decoder_channels, 3, padding="same", activation="swish", name="inpaint_stem"
    )(d)

    dilation_schedule = [1, 2, 4, 1, 2, 4, 1, 1][:decoder_blocks]
    for i, dilation in enumerate(dilation_schedule):
        shortcut = d
        h = layers.Conv2D(
            decoder_channels,
            3,
            padding="same",
            dilation_rate=dilation,
            activation="swish",
            name=f"inpaint_{i}_conv1",
        )(d)
        h = layers.Conv2D(
            decoder_channels,
            3,
            padding="same",
            name=f"inpaint_{i}_conv2",
        )(h)
        d = layers.Add(name=f"inpaint_{i}_add")([shortcut, h])
        d = layers.Activation("swish", name=f"inpaint_{i}_act")(d)

    fill_rgb = layers.Conv2D(
        3, 3, padding="same", activation="sigmoid", name="fill_rgb"
    )(d)

    known_mask_rgb = layers.Concatenate(name="known_mask_rgb")(
        [known_mask, known_mask, known_mask]
    )

    recon = layers.Lambda(
        lambda z: z[1] * z[0] + (1.0 - z[1]) * z[2],
        output_shape=(96, 96, 3),
        name="recon",
    )([known_canvas, known_mask_rgb, fill_rgb])

    model = keras.Model(patches, [recon, assign], name="neural_jigsaw_reconstructor")
    return model


model = build_jigsaw_reconstructor()
model.summary()

param_count = model.count_params()
print(f"Trainable parameters: {param_count:,}")
assert param_count < 6_000_000, "The model exceeds the 6M parameter limit."


## 4. Losses and training

`recon` uses MAE because this is also the required metric.  
`assign` uses sparse categorical cross-entropy as a small auxiliary loss to stabilize patch placement.


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "recon": keras.losses.MeanAbsoluteError(),
        "assign": keras.losses.SparseCategoricalCrossentropy(),
    },
    loss_weights={
        "recon": 1.0,
        "assign": 0.05,
    },
    metrics={
        "recon": [keras.metrics.MeanAbsoluteError(name="mae")],
        "assign": [keras.metrics.SparseCategoricalAccuracy(name="acc")],
    },
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_jigsaw_reconstructor.weights.h5",
        monitor="val_recon_mae",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_recon_mae",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_recon_mae",
        mode="min",
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
]

EPOCHS = 50

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
)


In [ ]:
# Load the best checkpoint, then save a final weights file.
model.load_weights("best_jigsaw_reconstructor.weights.h5")
model.save_weights("jigsaw_reconstructor.weights.h5")
print("Saved weights to jigsaw_reconstructor.weights.h5")


## 5. Optional: load weights through `gdown`

For the final submission, upload `jigsaw_reconstructor.weights.h5` to Google Drive, share it as **Anyone with the link**, copy the file ID, and paste it below.

This cell lets the notebook load weights in a fresh Colab runtime using `gdown`, as required by the assignment.


In [ ]:
# Paste the Google Drive file ID here after uploading your trained weights.
# Example from a Drive URL:
# https://drive.google.com/file/d/FILE_ID/view?usp=sharing
WEIGHTS_GDRIVE_FILE_ID = ""  # <-- fill this before final submission

if WEIGHTS_GDRIVE_FILE_ID:
    try:
        import gdown
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown

    gdown.download(
        id=WEIGHTS_GDRIVE_FILE_ID,
        output="jigsaw_reconstructor.weights.h5",
        quiet=False,
    )
    model.load_weights("jigsaw_reconstructor.weights.h5")
    print("Weights loaded successfully from gdown.")
else:
    print("No gdown file ID provided. Train the model or set WEIGHTS_GDRIVE_FILE_ID.")


## 6. Evaluation: MAE mean and standard deviation on the test set

The assignment asks for MAE over the test set and the standard deviation.  
The code below computes MAE per test image, then reports the mean and standard deviation across images.


In [ ]:
def predict_reconstruction(model, x, batch_size=None):
    pred = model.predict(x, batch_size=batch_size, verbose=0)
    if isinstance(pred, dict):
        pred = pred["recon"]
    elif isinstance(pred, (list, tuple)):
        pred = pred[0]
    return np.clip(pred, 0.0, 1.0)


def evaluate_mae(model, generator):
    all_mae = []

    for i in range(len(generator)):
        x, y_true = generator[i]
        y_pred = predict_reconstruction(model, x)

        per_image_mae = np.mean(np.abs(y_pred - y_true), axis=(1, 2, 3))
        all_mae.append(per_image_mae)

    all_mae = np.concatenate(all_mae, axis=0)
    return float(np.mean(all_mae)), float(np.std(all_mae)), all_mae


test_mae_mean, test_mae_std, test_mae_values = evaluate_mae(model, test_generator)
print(f"Test MAE mean: {test_mae_mean:.6f}")
print(f"Test MAE std:  {test_mae_std:.6f}")


## 7. Visual inspection

These plots are not part of the metric, but they help verify that the reconstruction is coherent and that missing borders are being filled smoothly.


In [ ]:
def show_reconstructions(model, generator, n=4):
    x, y_true = generator[0]
    y_pred = predict_reconstruction(model, x[:n])

    for i in range(n):
        plt.figure(figsize=(12, 4))

        plt.subplot(1, 3, 1)
        plot_canvas = np.zeros((96, 96, 3), dtype=np.float32)
        cell_dim = 32
        margin = 2
        for p in range(9):
            r, c = divmod(p, 3)
            plot_canvas[
                r * cell_dim + margin : r * cell_dim + margin + 28,
                c * cell_dim + margin : c * cell_dim + margin + 28,
            ] = x[i, p]
        plt.imshow(np.clip(plot_canvas, 0, 1))
        plt.title("Scrambled input order")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(y_pred[i])
        plt.title("Model reconstruction")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(y_true[i])
        plt.title("Ground truth")
        plt.axis("off")

        plt.show()


show_reconstructions(model, test_generator, n=4)


## 8. Baseline comparison

The baseline from the specification notebook repeats the mean patch and resizes it to 96×96.  
This is included only as a sanity check; the final reported score should be the model MAE above.


In [ ]:
def mean_patch_image(patches):
    patches = tf.convert_to_tensor(patches)
    b = tf.shape(patches)[0]
    mean_patch = tf.reduce_mean(patches, axis=1)
    mean_patches = tf.repeat(mean_patch[:, None, :, :, :], repeats=9, axis=1)

    out = tf.reshape(mean_patches, (b, 3, 3, 28, 28, 3))
    out = tf.transpose(out, [0, 1, 3, 2, 4, 5])
    out = tf.reshape(out, (b, 84, 84, 3))
    out = tf.image.resize(out, (96, 96))
    return out.numpy()


def evaluate_mean_patch_baseline(generator):
    all_mae = []
    for i in range(len(generator)):
        x, y_true = generator[i]
        y_pred = mean_patch_image(x)
        per_image_mae = np.mean(np.abs(y_pred - y_true), axis=(1, 2, 3))
        all_mae.append(per_image_mae)

    all_mae = np.concatenate(all_mae)
    return float(np.mean(all_mae)), float(np.std(all_mae))


baseline_mean, baseline_std = evaluate_mean_patch_baseline(test_generator)
print(f"Mean-patch baseline MAE mean: {baseline_mean:.6f}")
print(f"Mean-patch baseline MAE std:  {baseline_std:.6f}")
print(f"Neural model MAE mean:        {test_mae_mean:.6f}")
print(f"Neural model MAE std:         {test_mae_std:.6f}")


## Final submission checklist

Before submitting:

1. Restart the Colab runtime.
2. Run the notebook from top to bottom.
3. Verify that `Trainable parameters` is printed and remains below 6,000,000.
4. Verify that `WEIGHTS_GDRIVE_FILE_ID` downloads the weights with `gdown`.
5. Verify that `model.load_weights(...)` succeeds.
6. Verify that the test MAE mean and standard deviation are printed.
7. Submit this notebook as a single `.ipynb` file.
